# Practical 4 — Diabetes Prediction using KNN 🩺

**Dataset:** `diabetes.csv`  
**Aim:** Predict whether a patient belongs to the diabetes class (`Outcome`) using **K-Nearest Neighbors**, then calculate **confusion matrix, accuracy, error rate, precision and recall**.

**Beginner study guide:** Read the explanation cell, then run the code cell below it using **Shift + Enter**. Put `diabetes.csv` in the same working folder as this notebook.

> This is a classroom ML classification exercise, **not** a medical diagnostic tool.

## 1. What is KNN? (Theory)

**KNN = K-Nearest Neighbors.** It predicts a new patient's class by looking at the **K closest patients** in the training data.

**Example:** If `K=5`, look at the 5 nearest patients. If 3 have `Outcome=1` and 2 have `Outcome=0`, KNN predicts **1**.

- `Outcome = 0`: no diabetes class in this dataset
- `Outcome = 1`: diabetes class in this dataset
- **Classification** means predicting a category (0 or 1), not a price.
- **K=5** means consider five nearest neighbors.

**Memory trick:** *KNN = ask your 5 closest neighbors and take the majority vote.*

## 2. Import libraries

- **pandas** reads the CSV as a table.
- **train_test_split** separates training and testing rows.
- **StandardScaler** puts different features on a comparable scale, important because KNN uses distances.
- **KNeighborsClassifier** is our KNN algorithm.
- **metrics** measure prediction performance.

In [ ]:
# Import only the libraries we need
import pandas as pd                         # Work with tables / CSV files
from sklearn.model_selection import train_test_split  # Split data
from sklearn.preprocessing import StandardScaler      # Scale features
from sklearn.neighbors import KNeighborsClassifier    # KNN model
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_score, recall_score)  # Evaluation

## 3. Load and inspect the dataset

The uploaded file contains **768 rows and 9 columns**. Eight columns describe patient measurements, and the ninth (`Outcome`) is the answer.

| Column | Simple meaning |
|---|---|
| Pregnancies | Number of pregnancies |
| Glucose | Blood glucose measurement |
| BloodPressure | Blood pressure measurement |
| SkinThickness | Skinfold thickness |
| Insulin | Insulin measurement |
| BMI | Body mass index |
| DiabetesPedigreeFunction | Family-history-related numerical score |
| Age | Patient's age |
| Outcome | Class we want to predict: 0 or 1 |

`head()` displays the first five rows; `shape` gives (rows, columns).

In [ ]:
# Load the file (keep diabetes.csv in the notebook working folder)
df = pd.read_csv("diabetes.csv")

print("Rows and columns:", df.shape)  # Expected: (768, 9)
print(df.head())                       # Display first five rows
print("Missing values:", df.isnull().sum().sum())

## 4. Choose input (X) and output (y)

- **X** = questions / features used for prediction (all columns except `Outcome`).
- **y** = answer / target (`Outcome`).

`drop(columns=["Outcome"])` removes the answer from X so the model cannot simply read it.

In [ ]:
# X = inputs; y = correct answer
X = df.drop(columns=["Outcome"])  # All 8 measurement columns
y = df["Outcome"]                  # 0 or 1

## 5. Split into training and testing data

- **80% train:** data used to learn.
- **20% test:** unseen data used to check performance.
- `random_state=42` makes the split repeatable.
- `stratify=y` helps keep similar proportions of class 0 and class 1 in both sets.

In [ ]:
# Divide data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 6. Scale the features (very important for KNN)

Imagine one column has values around **150** (glucose) and another around **0.5** (family-history score). Without scaling, the larger numbers may dominate the distance calculation.

`StandardScaler` transforms features to have approximately **mean 0** and **standard deviation 1** on training data.

- `fit_transform(X_train)` = **learn** the scaling values from training data and **apply** them.
- `transform(X_test)` = apply **the same** scaling to testing data. **Never fit the scaler on test data.**

In [ ]:
# Scale inputs so no feature dominates just because its units are large
scaler = StandardScaler()                 # Create scaler
X_train = scaler.fit_transform(X_train)  # Learn scaling from training data
X_test = scaler.transform(X_test)        # Apply same scaling to test data

## 7. Train KNN and make predictions

- `n_neighbors=5` = look at five nearest training examples.
- `fit()` = store/learn from training examples.
- `predict()` = assign a 0 or 1 class to each test row.

In [ ]:
# Create, train, and use KNN
model = KNeighborsClassifier(n_neighbors=5)  # K = 5
model.fit(X_train, y_train)                   # Train model
pred = model.predict(X_test)                 # Predict on test data
print("First 10 predictions:", pred[:10])

## 8. Evaluate the model

**Confusion matrix** compares actual classes and predicted classes:

| | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | TN (correct negative) | FP (false alarm) |
| **Actual 1** | FN (missed positive) | TP (correct positive) |

- **Accuracy** = `(TP + TN) / all predictions` → how many were correct overall.
- **Error rate** = `1 - accuracy` → how many were wrong overall.
- **Precision** = `TP / (TP + FP)` → among predicted positives, how many were correct?
- **Recall** = `TP / (TP + FN)` → among actual positives, how many did we find?

**Memory trick:** Accuracy = overall correct; Precision = predicted positive correctness; Recall = found actual positives.

In [ ]:
# Compare predictions with correct answers
accuracy = accuracy_score(y_test, pred)  # Fraction of correct predictions

print("Confusion Matrix:")
print(confusion_matrix(y_test, pred))
print("Accuracy  :", accuracy)
print("Error Rate:", 1 - accuracy)       # Fraction of wrong predictions
print("Precision :", precision_score(y_test, pred, zero_division=0))
print("Recall    :", recall_score(y_test, pred, zero_division=0))

## 9. Optional: handling suspicious zeros (extra theory)

Your CSV has **no blank cells**, but some measurement columns contain `0` values (such as `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`). Some zeros may represent missing measurements rather than realistic readings.

Your original uploaded notebook replaces these selected zeros with missing values and fills them. The short exam code above follows the simpler version you provided. For a more careful ML workflow, replace suspicious zeros and calculate replacement values **from training data only**, to avoid data leakage. This is optional unless your examiner specifically asks for imputation.

## 10. Most important viva questions (read before exam)

1. **What is KNN?** A supervised classification algorithm that predicts from nearby training examples.
2. **What is K?** Number of nearest neighbors considered.
3. **Why choose K=5?** It is a simple starting value; other K values can be tested.
4. **What is supervised learning?** Learning using examples with known answers.
5. **What is classification?** Predicting a category such as 0 or 1.
6. **What is the target?** `Outcome`.
7. **What is X?** Eight input measurement columns.
8. **What is y?** The correct 0/1 outcome.
9. **Why scale?** KNN uses distances; different measurement units can distort those distances.
10. **What is StandardScaler?** It standardizes features using training mean and standard deviation.
11. **What does fit() do?** Trains/stores the model's training information.
12. **What does predict() do?** Predicts labels for new inputs.
13. **What is a confusion matrix?** A 2×2 table of correct and incorrect classifications.
14. **What is accuracy?** Fraction of all predictions that are correct.
15. **What is error rate?** Fraction of all predictions that are wrong.
16. **What is precision?** Fraction of predicted positives that are actually positive.
17. **What is recall?** Fraction of actual positives that were correctly detected.
18. **Why split data?** To evaluate the model on unseen examples.
19. **What is overfitting?** When a model performs well on training examples but poorly on new ones.
20. **Does KNN use distance?** Yes, usually Euclidean distance by default.

**Final memory sequence:** **Import → Read → X/y → Split → Scale → KNN → Predict → Metrics**.

## 11. Quick troubleshooting

- **FileNotFoundError** → put `diabetes.csv` in the VS Code notebook's working folder.
- **ModuleNotFoundError: sklearn** → run `%pip install scikit-learn pandas` in a separate notebook cell.
- **No kernel** → install the Microsoft Python and Jupyter extensions, choose a Python environment, and install `ipykernel`.
- **Wrong output after editing cells** → use **Run All** from the top so variables are created in the correct order.